# Lezione 02 — Statistica descrittiva

**Corso:** Statistica Applicata con Python · **Blocco 1:** Descrivere i dati
**Tempo stimato:** 100 minuti · **Capitolo del manuale:** `manual/02_estadistica_descriptiva_IT.md`

## Cosa vedrai in questa lezione

- Media, mediana e moda: cosa misura ciascuna e quando ingannano.
- Dispersione: varianza, deviazione standard (della popolazione e campionaria) e scarto interquartile (IQR).
- Quantili e percentili, e l'asimmetria.
- Valori anomali (*outlier*) con la regola dell'IQR e con lo z-score (qui solo come standardizzazione).
- Come scegliere il riassunto adatto quando ci sono valori estremi.

## Requisiti

- Lezione 01 (campione, popolazione, statistica campionaria, tipi di variabili) e NumPy/Pandas di base.
- Librerie: `numpy`, `pandas`, `matplotlib` (sono già installate in Google Colab).

> **Come usare questo notebook:** esegui le celle in ordine. Prima di ogni esperimento, scrivi (o pensa) cosa ti aspetti che accada; poi confronta con ciò che vedi.

---
## Cella 0 — Setup

Importiamo le librerie e fissiamo il **seme** (*seed*) del generatore casuale.

In [ ]:
# In Google Colab queste librerie sono già installate. In locale: pip install numpy pandas matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)          # seme fisso: otteniamo tutti gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerie caricate correttamente.")

---
## 1. I nostri dati: stipendi sintetici con valori estremi

Generiamo 300 stipendi mensili (in €) con una distribuzione asimmetrica, e **aggiungiamo a mano quattro valori molto alti**. Uno di essi (250 000 €) simula un errore di battitura: qualcuno ha scritto due zeri di troppo in uno stipendio di 2 500 €. *(Dati sintetici: esempio illustrativo.)*

In [ ]:
sueldos_base = np.round(rng.lognormal(mean=7.6, sigma=0.35, size=300), 2)   # stipendi "normali" (asimmetrici)
insertados = np.array([7_800.0, 9_500.0, 14_000.0, 250_000.0])                # valori estremi aggiunti a mano

sueldos = pd.Series(np.concatenate([sueldos_base, insertados]), name="sueldo")
print(f"n = {len(sueldos)}")
print(sueldos.head(8).to_string())

plt.hist(sueldos_base, bins=40, color="steelblue", edgecolor="white")
plt.title("Stipendi generati (senza i valori inseriti)")
plt.xlabel("Stipendio mensile (€)"); plt.ylabel("Dipendenti")
plt.show()

**Osserva:** la maggior parte guadagna tra 1 500 e 2 700 €, con una coda verso destra. Questo istogramma è solo un supporto visivo; nella lezione 3 vedremo in dettaglio come guardare una distribuzione.

---
## 2. Tendenza centrale: media, mediana e moda

Calcoliamo i tre riassunti sui 304 stipendi (i 300 normali più i 4 inseriti).

In [ ]:
print(f"Media   : {sueldos.mean():>10,.1f} €")
print(f"Mediana : {sueldos.median():>10,.1f} €")
print(f"Media dei 300 stipendi senza gli inseriti : {sueldos_base.mean():,.1f} €")
print(f"Mediana dei 300 stipendi senza gli inseriti: {np.median(sueldos_base):,.1f} €")

**Cosa osservi?** Confronta: aggiungendo solo 4 valori, di quanto si è spostata la media? E la mediana? Annotalo prima di proseguire.

*(La tua risposta qui.)*

### La moda con dati continui

Gli stipendi hanno i centesimi, quindi quasi nessun valore si ripete. Vediamo allora cosa significa la "moda".

In [ ]:
print("Valori distinti:", sueldos.nunique(), "su", len(sueldos))
print("Volte in cui si ripete il valore più frequente:", sueldos.value_counts().max())

# Soluzione ragionevole: raggruppare in intervalli da 500 € e cercare l'intervallo più frequente
intervalos = (sueldos // 500 * 500).astype(int)
print(intervalos.value_counts().sort_index().to_string())
print("Intervallo modale (inizia a):", intervalos.mode()[0], "€")

**Osserva:** con i dati "grezzi" la moda non significa nulla (tutti i valori compaiono una sola volta). Raggruppando in intervalli otteniamo invece un'idea di dove si concentrano gli stipendi, e questo è già quasi un istogramma.

---
## 3. Dispersione: varianza, deviazione standard e IQR

Misuriamo quanto si allontanano gli stipendi. Confrontiamo una misura che usa tutti i valori (deviazione standard) con una che usa solo il "50 % centrale" (IQR).

In [ ]:
def iqr(x):
    q1, q3 = np.quantile(x, [0.25, 0.75])
    return q3 - q1

print(f"Intervallo di variazione: {sueldos.max() - sueldos.min():>12,.1f} €")
print(f"Varianza campionaria    : {sueldos.var():>12,.1f} €²")
print(f"Deviazione standard     : {sueldos.std():>12,.1f} €")
print(f"IQR                     : {iqr(sueldos):>12,.1f} €")
print()
print(f"Senza i 4 inseriti -> deviazione standard: {sueldos_base.std(ddof=1):,.1f} €   IQR: {iqr(sueldos_base):,.1f} €")

**Cosa osservi?** Quale delle due misure di dispersione è cambiata di più aggiungendo i 4 valori estremi? Perché?

*(La tua risposta qui.)*

### Deviazione standard della popolazione e campionaria (`ddof`)

`numpy` e `pandas` calcolano la deviazione standard con **divisori diversi** per impostazione predefinita. Lo verifichiamo con un esempio che puoi controllare a mano: `[2, 4, 6]`. La media è 4, gli scarti sono −2, 0, 2, la somma dei quadrati è 8.

In [ ]:
mini = [2, 4, 6]
suma_cuadrados = sum((v - np.mean(mini)) ** 2 for v in mini)      # = 8

print("Somma dei quadrati          :", suma_cuadrados)
print("A mano, dividendo per n     :", round(np.sqrt(suma_cuadrados / 3), 3), "  (ddof=0)")
print("A mano, dividendo per n - 1 :", round(np.sqrt(suma_cuadrados / 2), 3), "  (ddof=1)")
print()
print("np.std(mini)               :", round(np.std(mini), 3), "  <- ddof=0 per impostazione predefinita")
print("pd.Series(mini).std()      :", round(pd.Series(mini).std(), 3), "  <- ddof=1 per impostazione predefinita")
print("np.std(mini, ddof=1)       :", round(np.std(mini, ddof=1), 3))
print()
# Conta con molti dati?
print(f"Con i 304 stipendi: np.std = {np.std(sueldos):.2f}   pandas std = {sueldos.std():.2f}   (quasi uguali)")

**Cosa osservi?** Con 3 dati la differenza è enorme (1,633 contro 2). Con 304 dati quasi sparisce. Quando analizzi un campione piccolo, scrivi sempre `ddof=1` in modo esplicito.

---
## 4. Quantili e percentili

I quantili dividono i dati ordinati. `pandas` usa per impostazione predefinita l'interpolazione lineare. Prima verifichiamo a mano con i 9 stipendi del capitolo, dove le posizioni sono esatte (0,25 · 8 = 2, 0,5 · 8 = 4, 0,75 · 8 = 6).

In [ ]:
nueve = pd.Series([1600, 1800, 1900, 2000, 2100, 2200, 2500, 2800, 9500])
print(nueve.quantile([0.25, 0.5, 0.75]).to_string())
print("Terzo, quinto e settimo dato ordinati:", nueve.sort_values().iloc[[2, 4, 6]].tolist())

In [ ]:
percentiles = sueldos.quantile([0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(1)
percentiles.index = [f"P{int(p * 100)}" for p in percentiles.index]
print(percentiles.to_string())
print()
print(sueldos.describe().round(1).to_string())

**Osserva:** `describe()` fornisce in un colpo n, media, deviazione standard, minimo, quartili e massimo. Confronta la media (`mean`) con la mediana (`50%`) e il massimo con il 75° percentile: si intuisce già che c'è asimmetria e ci sono valori estremi. Ciò che `describe()` **non** fa è segnalare gli anomali o misurare l'asimmetria.

---
## 5. Asimmetria

Confrontiamo gli stipendi con un campione simmetrico di riferimento (altezze sintetiche in cm, anch'esse *esempio illustrativo*). Guardiamo la differenza media − mediana, il coefficiente `skew()` e un istogramma.

In [ ]:
alturas = pd.Series(rng.normal(170, 8, size=304), name="altura")     # campione simmetrico di riferimento

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for eje, serie, titulo, unidad in [(ejes[0], sueldos, "Stipendi (asimmetrica)", "€"),
                                   (ejes[1], alturas, "Altezze (simmetrica)", "cm")]:
    eje.hist(serie, bins=50, color="lightsteelblue", edgecolor="white")
    eje.axvline(serie.mean(), color="crimson", linestyle="--", label=f"media = {serie.mean():,.0f}")
    eje.axvline(serie.median(), color="black", linestyle="-", label=f"mediana = {serie.median():,.0f}")
    eje.set_title(titulo); eje.set_xlabel(unidad); eje.legend()
plt.show()

print(f"Stipendi: media − mediana = {sueldos.mean() - sueldos.median():+.1f} €   skew = {sueldos.skew():.2f}")
print(f"Altezze: media − mediana = {alturas.mean() - alturas.median():+.2f} cm   skew = {alturas.skew():.2f}")

**Cosa osservi?** Nella distribuzione simmetrica media e mediana quasi coincidono e `skew()` è vicino a 0. Negli stipendi la media sta sopra la mediana e `skew()` è chiaramente positivo: la coda lunga a destra trascina la media.

---
## 6. Valori anomali: regola dell'IQR e z-score

Applichiamo le due regole ai 304 stipendi. **Ricorda:** in questa lezione lo z-score è solo una **standardizzazione** (`(x − media) / deviazione standard`); il suo legame con la distribuzione normale si vede nella lezione 5.

In [ ]:
q1, q3 = sueldos.quantile([0.25, 0.75])
lim_inf, lim_sup = q1 - 1.5 * iqr(sueldos), q3 + 1.5 * iqr(sueldos)
atipicos_iqr = sueldos[(sueldos < lim_inf) | (sueldos > lim_sup)]

z = (sueldos - sueldos.mean()) / sueldos.std()          # standardizzazione: media 0, deviazione standard 1
atipicos_z = sueldos[z.abs() > 3]

print(f"Limiti della regola dell'IQR: [{lim_inf:,.0f} €, {lim_sup:,.0f} €]")
print(f"Anomali secondo l'IQR    : {len(atipicos_iqr)}")
print(f"Anomali secondo |z| > 3: {len(atipicos_z)} -> {sorted(atipicos_z.round(0).tolist())}")
print(f"z-score media = {z.mean():.2f}, deviazione standard = {z.std():.2f}")
print("I 4 stipendi inseriti a mano, sono individuati dall'IQR?", [bool(v in set(atipicos_iqr)) for v in insertados])
print("I 4 stipendi inseriti a mano, sono individuati da z?     ", [bool(v in set(atipicos_z)) for v in insertados])

**Cosa osservi?** La regola dell'IQR segnala molti più valori di |z| > 3. Perché? Pensa a cosa succede a media e deviazione standard quando c'è un valore da 250 000 €.

*(La tua risposta qui.)*

### Lo z-score con pochi dati: l'esempio dei 9 stipendi

In [ ]:
z_nueve = (nueve - nueve.mean()) / nueve.std()
print("z-score:", z_nueve.round(2).tolist())
print("z massimo possibile con n = 9:", round((len(nueve) - 1) / np.sqrt(len(nueve)), 2))

q1n, q3n = nueve.quantile([0.25, 0.75])
print("Limiti IQR:", (q1n - 1.5 * (q3n - q1n), q3n + 1.5 * (q3n - q1n)), "->", nueve[(nueve < q1n - 1.5 * (q3n - q1n)) | (nueve > q3n + 1.5 * (q3n - q1n))].tolist())

**Osserva:** il 9 500 ha z = 2,64: non supera 3, quindi |z| > 3 **non lo individua**, mentre la regola dell'IQR sì. Con *n* = 9 è impossibile superare z = 2,67. L'anomalo stesso contamina media e deviazione standard.

---
## 7. Esperimento guidato: aggiungere valori estremi uno alla volta

Partiamo dai 300 stipendi "normali" e aggiungiamo progressivamente valori sempre più estremi: 8 000, 15 000, 30 000, 60 000, 120 000 e 250 000 €. A ogni passo calcoliamo due riassunti di centro (media e mediana) e due di dispersione (deviazione standard e IQR). Quali si muovono e quali no?

**Prima di eseguire, prevedi:** cosa accadrà alla media dopo l'aggiunta del sesto valore? E alla mediana? E all'IQR?

In [ ]:
extremos = [8_000, 15_000, 30_000, 60_000, 120_000, 250_000]

filas = []
for k in range(len(extremos) + 1):
    muestra_k = np.concatenate([sueldos_base, extremos[:k]])        # k = numero di estremi aggiunti
    filas.append({"k": k,
                  "media": muestra_k.mean(),
                  "mediana": np.median(muestra_k),
                  "desv_std": muestra_k.std(ddof=1),
                  "iqr": iqr(muestra_k)})

evolucion = pd.DataFrame(filas).set_index("k").round(1)
display(evolucion)

cambio = (evolucion.loc[len(extremos)] / evolucion.loc[0] - 1)
print("Variazione relativa tra 0 e 6 estremi aggiunti:")
print(cambio.map("{:+.0%}".format).to_string())

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].plot(evolucion.index, evolucion["media"], "o-", color="crimson", label="media")
ejes[0].plot(evolucion.index, evolucion["mediana"], "s-", color="black", label="mediana")
ejes[0].set_title("Centro"); ejes[0].set_xlabel("Valori estremi aggiunti"); ejes[0].set_ylabel("€"); ejes[0].legend()
ejes[1].plot(evolucion.index, evolucion["desv_std"], "o-", color="crimson", label="deviazione standard")
ejes[1].plot(evolucion.index, evolucion["iqr"], "s-", color="black", label="IQR")
ejes[1].set_title("Dispersione"); ejes[1].set_xlabel("Valori estremi aggiunti"); ejes[1].set_ylabel("€"); ejes[1].legend()
plt.show()

**Cosa osservi?** Annota le tue conclusioni:

- Di quanto è cresciuta la media e di quanto la mediana tra k = 0 e k = 6? Coincide con la tua previsione?
- Quale delle due misure di dispersione esplode e quale non si muove quasi?
- Se dovessi riportare lo "stipendio tipico" e la sua dispersione dopo aver aggiunto i 6 estremi, quali due cifre useresti?

*(La tua risposta qui.)*

**Conclusione:** media e deviazione standard usano tutti i valori e si lasciano trascinare dagli estremi; mediana e IQR dipendono dall'ordine dei dati e restano quasi impassibili. Quando ci sono valori estremi, i riassunti robusti raccontano meglio la storia della maggior parte dei dati.

---
## 8. E cosa facciamo con gli anomali? Indagare prima di eliminare

Individuare un anomalo non autorizza a cancellarlo. Guardiamo i valori segnalati dalla regola dell'IQR che stanno sopra 7 000 € e confrontiamo tre versioni dei dati.

In [ ]:
print("Valori molto alti segnalati dall'IQR:", sorted(atipicos_iqr[atipicos_iqr > 7_000].tolist()))

# Supponiamo di aver verificato con le Risorse Umane: 250 000 era un errore di battitura (doveva essere 2 500); gli altri tre sono stipendi reali.
sueldos_corregidos = sueldos.replace(250_000.0, 2_500.0)

def resumen(x):
    """Riassunto robusto di un campione numerico."""
    return pd.Series({"n": len(x), "media": x.mean(), "mediana": x.median(),
                      "desv_std": x.std(ddof=1), "iqr": iqr(x), "asimetria": x.skew()})

comparacion = pd.DataFrame({
    "con l'errore": resumen(sueldos),
    "errore corretto": resumen(sueldos_corregidos),
    "senza i 4 inseriti": resumen(pd.Series(sueldos_base)),
}).round(1)
display(comparacion)

**Cosa osservi?** Correggere l'errore di battitura (un dato verificato) cambia molto media e deviazione standard; i tre stipendi reali molto alti continuano a trascinare la media verso l'alto, ma non vanno cancellati: sono persone reali. Mediana e IQR sono quasi uguali nelle tre versioni.

> **Regola pratica:** correggi ciò che puoi verificare come errore; conserva ciò che è reale e scegli riassunti robusti; e documenta sempre cosa hai fatto.

---
## Riepilogo della lezione

1. **Centro:** media (usa tutto, sensibile agli estremi), mediana (robusta), moda (solo con categorie o dati discreti).
2. **Dispersione:** deviazione standard (sensibile) e IQR (robusto). `np.std` usa `ddof=0`, `pandas` usa `ddof=1`: decidi e scrivilo.
3. **Quantili e percentili:** il quantile *p* lascia sotto di sé una frazione *p* dei dati; `describe()` è un buon primo sguardo.
4. **Asimmetria:** se media > mediana e `skew()` > 0, c'è una coda a destra.
5. **Anomali:** regola dell'IQR (robusta) e z-score (qui solo standardizzazione; con pochi dati può non individuarli). Indaga prima di eliminare.
6. **Scegliere il riassunto:** simmetrico senza estremi -> media e deviazione standard; asimmetrico o con estremi -> mediana e IQR. E sempre, guarda i dati.

**Prossimo passo:** Lezione 3 — Distribuzioni empiriche (istogrammi, ECDF e altri modi di guardare una distribuzione). E ora, a esercitarti: `notebooks_practica/clase02_practica_estadistica_descriptiva_IT.ipynb`.